<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/07_export_mobile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 · Export for phones (TFLite, Core ML, Flutter)

Turns a trained run from notebook 05 into phone models. Each model is checked on the test
split against the original `.pt`.

```
reads   MyDrive/clear_vision/exports/<RUN_NAME>/best.pt (+ metrics.json for the image size)
        MyDrive/clear_vision/datasets/merged.zip + merged.done   (prepare_dataset, unpacked to /content)
writes  MyDrive/clear_vision/exports/<RUN_NAME>/mobile/
          clear_vision_<RUN_NAME>_int8.tflite      TFLite INT8, calibrated on 200 train images
          clear_vision_<RUN_NAME>_fp16.tflite      TFLite FP16
          clear_vision_<RUN_NAME>.mlpackage        Core ML FP16 with NMS   (Flutter plugin, iOS bundle)
          clear_vision_<RUN_NAME>.mlpackage.zip    the same, zipped        (Flutter plugin, assets/models/)
          clear_vision_<RUN_NAME>.tflite           LiteRT FP32, no NMS     (Flutter plugin, Android)
          labels.txt, thresholds.json, mobile_metrics.json
```

- **Any run:** change only `RUN_NAME` (`budget_v1`, `v2`, ...). The image size comes from
  that run's `metrics.json`; budget_v1 was trained at 512.
- **Flutter plugin (`ultralytics_yolo`, its `doc/models.md`):**
  - Android: `format="litert"` with `nms=None`; the plugin runs NMS. FP32 is one of the
    two precisions it recommends.
  - iOS: a Core ML `.mlpackage`, or `.mlpackage.zip` for `assets/models/`.
  - The plugin reads task and class names from the metadata Ultralytics embeds, which
    this notebook checks.
- **Formats in Ultralytics 8.4:** TFLite is `format="litert"` and precision is
  `quantize=8/16`. LiteRT export has no FP16, so the FP16 `.tflite` comes from the
  TensorFlow export (`*_float16.tflite`).
  - That file is NHWC with pixel coordinates; the LiteRT ones are NCHW with normalized
    coordinates.
- **Core ML's embedded NMS keeps boxes from 0.15,** the lowest per-class threshold, so the
  0.15 thresholds for curb, stairs, pothole, pole and manhole still work on iPhone.
- **One process per step:** each export and each validation runs as its own Python
  process. Exports install packages (other numpy and torch versions) that break modules
  already loaded.
- **Validation:** every model goes through notebook 08's evaluator. mAP50 is COCO-style AP
  at IoU 0.5; recall is measured at each class's own threshold from `thresholds.json`.
  Ultralytics 8.4's own LiteRT reader can't score the TensorFlow-route FP16 file, so
  `.tflite` files are read with `scripts/mobile_export.py` `TFLiteDetector`.
- **Core ML runs only on macOS,** so in Colab it is exported and its metadata checked, but
  its mAP is not measured. Run sections 2 and 5 on a Mac to measure it.
- **Images are never read from Drive.** Exports are built under `/content` and only the
  finished files are copied to Drive.
- **Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU). Exports run on the CPU, so
  any runtime works, but the `.pt` validation is faster on a GPU. Run all cells.

In [ ]:
# Settings
RUN_NAME = "budget_v1"  # the only thing to change for a later model (v2, v3, ...)
IMGSZ = None  # None = the training size from exports/<RUN_NAME>/metrics.json (budget_v1: 512)
CALIB_IMAGES = 200  # train images for INT8 calibration
SEED = 42
MAX_FP16_LOSS = 3.0  # % mAP50 loss allowed for the FP16 models

REPO_URL = "https://github.com/Zikirullo/clear-vision.git"
BRANCH = "develop"
DRIVE_ROOT = "/content/drive/MyDrive/clear_vision"
MERGED_ZIP = f"{DRIVE_ROOT}/datasets/merged.zip"
LOCAL_DATASET = "/content/datasets/merged"
WORK = f"/content/export_{RUN_NAME}"  # local build folder; only finished files go to Drive

## 1. Setup

In [ ]:
# 1. Setup: imports, platform, Drive, repo, packages, the run's files
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
import yaml

# Not Path("/kaggle").exists(): Colab runtimes have that folder too. Same rule as
# src/colab_data.py is_kaggle(); needed before the repo is cloned (checked below).
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) and not (
    "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ)
print(f"Platform: {'Kaggle' if ON_KAGGLE else 'Colab'}")
if ON_KAGGLE:
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
else:
    from google.colab import drive

    drive.mount("/content/drive")
    if not Path(DRIVE_ROOT).exists():
        raise RuntimeError(f"Google Drive is not connected: {DRIVE_ROOT} does not exist. Re-run this cell and "
                           "allow access to Google Drive, and check that DRIVE_ROOT in Settings is right.")

REPO_DIR = Path("/kaggle/working/clear-vision" if ON_KAGGLE else "/content/clear-vision")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print(f"Repo at {REPO_DIR}: {commit.strip()}")

%pip install -q ultralytics

sys.path.insert(0, str(REPO_DIR))
from ultralytics import YOLO  # noqa: E402

import scripts.mobile_export as me  # noqa: E402
import scripts.train_utils as tu  # noqa: E402
from src.colab_data import is_kaggle, prepare_dataset  # noqa: E402

assert ON_KAGGLE == is_kaggle(), "platform check differs from src/colab_data.py is_kaggle()"
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else
      "No GPU: exports work, validation of the .pt is slower.")

RUN_DIR = Path(DRIVE_ROOT) / "exports" / RUN_NAME
BEST_DRIVE = RUN_DIR / "best.pt"
if not BEST_DRIVE.exists():
    raise RuntimeError(f"{BEST_DRIVE} not found: check RUN_NAME, or run notebook 05 (section 7 exports best.pt).")
MOBILE = RUN_DIR / "mobile"
IMGSZ = IMGSZ or me.read_imgsz(RUN_DIR / "metrics.json")
WORK = Path(WORK)
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
BEST = WORK / "best.pt"  # exports are written next to the weights: keep them off Drive
shutil.copyfile(BEST_DRIVE, BEST)
free = tu.check_space(MOBILE, 200 * 10**6)
print(f"Run {RUN_NAME}: {BEST_DRIVE} ({me.size_mb(BEST):.1f} MB), imgsz {IMGSZ}, Drive free {free / 1e9:.1f} GB")

## 2. Data

Unpacks the dataset to `/content` and checks that the model was trained on the same 20
classes. It also picks 200 random train images for INT8 calibration.

In [ ]:
# 2. Unpack, check classes, calibration set, labels.txt + thresholds.json
DATA_YAML = prepare_dataset(MERGED_ZIP, LOCAL_DATASET)
DATASET = DATA_YAML.parent
CLASSES = list(json.loads(Path(MERGED_ZIP).with_name("merge_report.json").read_text())["classes"])
model_names = list(YOLO(str(BEST)).names.values())
if model_names != CLASSES:
    raise RuntimeError(f"{RUN_NAME} was trained on other classes than merged.zip:\n{model_names}\nvs\n{CLASSES}")
CALIB_YAML = me.calibration_yaml(DATASET, CLASSES, CALIB_IMAGES, SEED)

OUT = WORK / "mobile"
OUT.mkdir(exist_ok=True)
cfg = yaml.safe_load((REPO_DIR / "configs" / "default.yaml").read_text())
DEFAULT_THR, THRESHOLDS = me.thresholds(cfg, CLASSES)
me.write_side_files(OUT, CLASSES, DEFAULT_THR, THRESHOLDS, RUN_NAME, IMGSZ)
MIN_THR = min(THRESHOLDS.values())
n_calib = len((DATASET / "calib_train.txt").read_text().splitlines())
print(f"{len(CLASSES)} classes match. Calibration: {n_calib} train images. "
      f"Thresholds: {DEFAULT_THR} default, lowest {MIN_THR}")

## 3. Export

- Each export runs as its own Python process
  (`python scripts/mobile_export.py export ...`), starts from `best.pt`, and is renamed
  right away (LiteRT always writes `best.tflite`).
- The first export installs the export packages, which takes a few minutes.
- If one export fails, its error is shown and the others still run.

In [ ]:
# 3. Export TFLite INT8 / FP16, Core ML FP16 (+ zip), Flutter LiteRT: one process each
NAME = f"clear_vision_{RUN_NAME}"
TOOL = REPO_DIR / "scripts" / "mobile_export.py"
ARTIFACTS: dict[str, Path] = {}
ERRORS: dict[str, str] = {}


def run_step(*args):
    """Run scripts/mobile_export.py in a fresh process; return its RESULT json."""
    proc = subprocess.run([sys.executable, str(TOOL), *map(str, args)], cwd=WORK, capture_output=True, text=True)
    result = next((ln for ln in reversed(proc.stdout.splitlines()) if ln.startswith("RESULT ")), None)
    if proc.returncode or result is None:
        raise RuntimeError("\n".join((proc.stdout + proc.stderr).splitlines()[-25:]))
    return json.loads(result[len("RESULT "):])


def export(label, out_name, fmt, *options):
    print(f"=== {label}: format={fmt} {' '.join(options)}", flush=True)
    try:
        info = run_step("export", BEST, fmt, OUT / out_name, *options, "--imgsz", IMGSZ)
        ARTIFACTS[label] = Path(info["path"])
        print(f"-> {out_name} ({info['size_mb']} MB)")
    except RuntimeError as e:
        ERRORS[label] = str(e).splitlines()[-1]
        print(f"FAILED {label}:\n{e}")


export("TFLite INT8", f"{NAME}_int8.tflite", "litert", "quantize=8", f"data={CALIB_YAML}")
export("TFLite FP16", f"{NAME}_fp16.tflite", "saved_model")
export("Core ML FP16 (NMS)", f"{NAME}.mlpackage", "coreml", "nms=true", "quantize=16", f"conf={MIN_THR}")
export("Flutter Android LiteRT", f"{NAME}.tflite", "litert", "nms=null")
if "Core ML FP16 (NMS)" in ARTIFACTS:
    ARTIFACTS["Flutter iOS zip"] = me.zip_mlpackage(ARTIFACTS["Core ML FP16 (NMS)"], OUT / f"{NAME}.mlpackage.zip")

print("\nExported:", ", ".join(p.name for p in ARTIFACTS.values()))
if ERRORS:
    print("FAILED:", json.dumps(ERRORS, indent=2))

## 4. Embedded metadata

The Flutter plugin infers the task and class names from this metadata. Every model must
say `detect` and list our 20 classes in order.

In [ ]:
# 4. Check the metadata the plugin reads
META = {}
for label, path in ARTIFACTS.items():
    if path.suffix == ".tflite":
        meta = me.tflite_metadata(path)
    elif path.suffix == ".mlpackage":
        meta = me.coreml_metadata(path)
    else:
        continue
    names = me.names_from_metadata(meta)
    ok = bool(meta) and meta.get("task") == "detect" and names == CLASSES
    META[label] = {"task": (meta or {}).get("task"), "classes_ok": names == CLASSES, "imgsz": (meta or {}).get("imgsz")}
    print(f"{'OK  ' if ok else 'BAD '} {path.name:40s} task={META[label]['task']} classes={'20 in order' if names == CLASSES else names}")

## 5. Validate on the test split

- Each model is scored on the local test images, in its own process, with notebook 08's
  evaluator: mAP50, and recall for the critical classes at their own thresholds (what the
  app would report).
- The FP16 limit is `MAX_FP16_LOSS` % mAP50 below the `.pt`.
- Core ML is scored only on macOS.

In [ ]:
# 5. mAP50, critical-class recall and size vs the original .pt
THR_FILE = OUT / "thresholds.json"
EVALS = {}
for label, path in [("PyTorch (.pt)", BEST)] + [(k, v) for k, v in ARTIFACTS.items() if v.suffix != ".zip"]:
    print(f"validating {label} ...", flush=True)
    try:
        EVALS[label] = run_step("evaluate", path, DATA_YAML, "--imgsz", IMGSZ, "--thresholds", THR_FILE)
    except RuntimeError as e:
        EVALS[label] = {"error": str(e).splitlines()[-1]}
        print(f"  failed: {EVALS[label]['error']}")

BASE_MAP = EVALS["PyTorch (.pt)"]["map50"]
base_mb = me.size_mb(BEST)
ROWS = []
for label, result in EVALS.items():
    path = BEST if label == "PyTorch (.pt)" else ARTIFACTS[label]
    note = " (macOS only)" if "skipped" in result else " (failed)" if "error" in result else ""
    ROWS.append(me.result_row(label + note, path, result.get("recall"), result.get("map50"), BASE_MAP, base_mb))

print(me.format_rows(ROWS))
for r in ROWS:
    if "FP16" in r["model"] and r["mAP50 loss %"] is not None:
        verdict = "PASS" if r["mAP50 loss %"] < MAX_FP16_LOSS else "FAIL"
        print(f"{verdict}: {r['model']} loses {r['mAP50 loss %']:.2f}% mAP50 (limit {MAX_FP16_LOSS}%)")
display(pd.DataFrame(ROWS).set_index("model"))

## 6. Save to Drive

In [ ]:
# 6. Copy the models, labels.txt, thresholds.json and mobile_metrics.json to exports/<RUN_NAME>/mobile/
(OUT / "mobile_metrics.json").write_text(json.dumps({
    "run": RUN_NAME, "imgsz": IMGSZ, "commit": commit.strip(), "calibration_images": CALIB_IMAGES,
    "metric": "notebook 08 evaluator: AP50 (101-point) on the test split; recall at per-class thresholds",
    "base_map50": BASE_MAP, "rows": ROWS, "evaluations": EVALS, "metadata": META, "errors": ERRORS,
}, indent=2))

MOBILE.mkdir(parents=True, exist_ok=True)
for item in sorted(OUT.iterdir()):
    me.copy_artifact(item, MOBILE / item.name)
for item in sorted(MOBILE.iterdir()):
    print(f"{item.name:42s} {me.size_mb(item):8.2f} MB")

## Using the files

**Flutter (`ultralytics_yolo`)**
- **Android:** put `clear_vision_<RUN_NAME>.tflite` in `assets/models/`, or in
  `android/app/src/main/assets`.
- **iOS:** put `clear_vision_<RUN_NAME>.mlpackage.zip` in `assets/models/`, or drag the
  `.mlpackage` into `ios/Runner.xcworkspace` and load it by name.
- **In Dart:** `YOLO(modelPath: 'assets/models/clear_vision_budget_v1.tflite', task: YOLOTask.detect)`.
  The task and class names come from the embedded metadata. `YOLO.inspectModel(path)` shows
  them.
- **Thresholds:** apply `thresholds.json` (per-class minimum confidence) to the results in
  the app; the plugin uses one global threshold.

**Other apps:** `_int8.tflite` is the smallest and fastest on Android CPUs and NPUs.
`_fp16.tflite` keeps the accuracy (GPU delegate). `labels.txt` lists the class names in ID
order.